# 🧪 GIADA Task 3d — matrice causale m+h
Otto bracci shared × tre seed in parallelo GPU, più controllo independent vettorializzato. Nessun fresh Task 3c viene letto.

In [ ]:
from pathlib import Path
import base64, json, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_3d'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 3d richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateGeneralizationDiagnosisConfig,prepare_joint_gate_dataset,prepare_joint_gate_generalization_diagnosis,run_joint_gate_generalization_diagnosis
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_generalization_diagnosis_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),'preregistration':prereg})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_generalization_diagnosis')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
base=prepare_joint_gate_dataset(formula); config=JointGateGeneralizationDiagnosisConfig()
bundle=prepare_joint_gate_generalization_diagnosis(formula,base,config)
display({'contract':bundle['contract'],'parallel_shared_candidates':len(config.shared_arms)*len(config.seeds),'parallel_independent_candidates':len(config.seeds),'parallel_wide_candidates':len(config.seeds),'checkpoints':config.checkpoints})
assert not bundle['contract']['fresh_task3c_accessed'] and not bundle['contract']['fresh_used_for_selection']


## ⚡ Training multifattoriale GPU
Il progresso è stampato solo ai checkpoint; nessun array esteso viene inviato al browser.

In [ ]:
report=run_joint_gate_generalization_diagnosis(bundle,OUTPUT_DIR,config,code_revision=REVISION)
display({'valid':report['valid'],'execution':report['execution'],'effects':report['causal_effect_fractions'],'supported_causes':report['supported_causes'],'best_development_arm':report['best_development_arm'],'fresh_task3c_accessed':report['fresh_task3c_accessed'],'task4_authorized':report['task4_authorized'],'next_step':report['next_step']})
assert report['valid'] and not report['fresh_task3c_accessed'] and not report['task4_authorized']


## 📦 Download Blob/base64
Metodo stabile del progetto per scaricare lo ZIP da Kaggle.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_generalization_diagnosis','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
